In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import itertools
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


import unicodedata
import nltk

from nltk.stem import WordNetLemmatizer

from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import LinearSVC


# 配置图表风格
sns.set_theme(style="whitegrid")

In [3]:
# 读取训练集和测试集
train_df = pd.read_json('data/train.json')
test_df = pd.read_json('data/test.json')

# 合并训练集与测试集的特征列
data_df = pd.concat([train_df['ingredients'], test_df['ingredients']], keys=['train', 'test'])
all_ingredients = [item for sublist in data_df for item in sublist]


In [4]:
print("--- 统计词频 ---")

word_counts = Counter(all_ingredients)
print(f"食材数据量为 {len(all_ingredients)}")
print(f"食材种类数为 {len(word_counts)}")

--- 统计词频 ---
食材数据量为 428275
食材种类数为 6714


In [5]:
print("--- 寻找短词 ---")

# 找出长度 <= 3 的词
short_words = [word for word in word_counts.keys() if len(word) <= 3]

# 按出现次数(word_counts)倒序排列
short_words_sorted = sorted(short_words, key=lambda x: word_counts[x], reverse=True)

print(f"总共找到 {len(short_words)} 个短词, 出现次数如下(格式: 词 -> 出现次数):")
for w in short_words_sorted:
    print(f"'{w}': {word_counts[w]}", end="  |  ")

--- 寻找短词 ---
总共找到 32 个短词, 出现次数如下(格式: 词 -> 出现次数):
'oil': 1970  |  'ham': 222  |  'ice': 111  |  'fat': 92  |  'rum': 77  |  'rib': 59  |  'cod': 36  |  'msg': 29  |  'jam': 27  |  'rub': 17  |  'pig': 15  |  'min': 14  |  'gin': 14  |  'dal': 14  |  'ale': 13  |  'soy': 12  |  'red': 10  |  'ahi': 5  |  'rye': 4  |  'mie': 3  |  'roe': 3  |  'eel': 2  |  'mi': 2  |  'any': 2  |  'tvp': 2  |  'v8': 2  |  'poi': 1  |  'kha': 1  |  'uni': 1  |  'hen': 1  |  'lox': 1  |  'val': 1  |  

In [6]:
print("--- 观察包含除字母和空格之外字符的配料 ---")

noise_pattern = re.compile(r'[^a-zA-Z\s]')
noisy_ingredients = [word for word in word_counts.keys() if noise_pattern.search(word)]

print(f"发现 {len(noisy_ingredients)} 个配料:")
print(noisy_ingredients)

# 发现以下几类情况:
# 食材的状态在逗号后 或通过连字符表示 或直接用相关形容词
# 数字, /, %, (), ., 单位等表示食材的规格
# 品牌名, 通用营销词
# 非标准ascii字符或其他特殊字符

--- 观察包含除字母和空格之外字符的配料 ---
发现 455 个配料:
['non-fat sour cream', 'whole kernel corn, drain', 'all-purpose flour', 'red bell pepper, sliced', 'soft-wheat flour', 'green bell pepper, slice', 'tomato purée', 'extra-virgin olive oil', 'half & half', 'hard-boiled egg', 'stone-ground cornmeal', 'part-skim mozzarella', 'parmigiano-reggiano cheese', '1% low-fat milk', 'quick-cooking oats', 'gluten-free pasta', 'Diamond Crystal® Kosher Salt', 'bow-tie pasta', 'low-fat mozzarella cheese', 'low-fat ricotta cheese', 'cream cheese, soften', 'long-grain rice', 'crème fraîche', 'low-fat cottage cheese', 'black-eyed peas', 'part-skim mozzarella cheese', 'part-skim ricotta cheese', 'boneless, skinless chicken breast', 'low-fat milk', 'lasagna noodles, cooked and drained', 'fat-free chicken broth', 'uncook medium shrimp, peel and devein', "soft goat's cheese", 'frozen whip topping, thaw', 'roast red peppers, drain', 'low-fat greek yogurt', 'chicken-flavored soup powder', 'plain low-fat yogurt', 'short-grain

In [7]:
# 抓取所有带逗号的食材
comma_ingredients = []
for items in train_df['ingredients']:
    for item in items:
        if ',' in item:
            comma_ingredients.append(item)

# 2. 统计总数
total_comma = len(comma_ingredients)
print(f"总共有 {total_comma} 个食材包含逗号。")

# 3. 抽样观察 (看前 50 个)
print("\n--- 随机抽样 20 个带逗号的样本 ---")
# 为了由代表性，我们看一些不同的例子
unique_comma_samples = list(set(comma_ingredients))
for sample in unique_comma_samples[:20]:
    print(f"[{sample}]")

# 4. 深度分析：逗号后面到底是啥？
# 我们尝试提取逗号后面的单词，统计一下频率
after_comma_words = []
for item in comma_ingredients:
    # 分割，取逗号后的部分，去除首尾空格
    parts = item.split(',')
    if len(parts) > 1:
        # 取逗号后第一个单词 (比如 "sliced" from "red pepper, sliced")
        modifier = parts[1].strip().split(' ')[0].lower()
        after_comma_words.append(modifier)

modifier_counts = Counter(after_comma_words)

print("\n--- 逗号后面最常出现的 30 个词 ---")
print("(如果这些词全是动作/形容词，说明'一刀切'是安全的)")
for word, count in modifier_counts.most_common(30):
    print(f"{word}: {count}")

总共有 496 个食材包含逗号。

--- 随机抽样 20 个带逗号的样本 ---
[bacon, crisp-cooked and crumbled]
[sweet italian sausag links, cut into]
[frozen chopped spinach, thawed and squeezed dry]
[clams, well scrub]
[rotel pasta, cook and drain]
[dri thyme leaves, crush]
[green bell pepper, slice]
[clove garlic, fine chop]
[dri basil leaves, crush]
[linguine, cook and drain]
[fresh spinach leaves, rins and pat dry]
[frozen whip topping, thaw]
[mussels, well scrubbed]
[rotini pasta, cook and drain]
[frozen crabmeat, thaw and drain]
[frozen mixed thawed vegetables,]
[small capers, rins and drain]
[lasagna noodles, cooked and drained]
[english muffins, split and toasted]
[frozen orange juice concentrate, thawed and undiluted]

--- 逗号后面最常出现的 30 个词 ---
(如果这些词全是动作/形容词，说明'一刀切'是安全的)
drain: 121
soften: 111
slice: 35
skinless: 30
cooked: 30
rins: 19
cook: 19
crush: 18
thawed: 18
peel: 17
fine: 16
cut: 15
well: 10
drained: 9
thaw: 6
sliced: 5
undrain: 5
: 4
split: 3
crisp-cooked: 3
1: 1
wine: 1


In [8]:
# 下载语料库
try:
    nltk.data.find('corpora/wordnet.zip')
except LookupError:
    print("正在下载 NLTK 数据...")
    nltk.download('wordnet')
    nltk.download('omw-1.4')

lemma = WordNetLemmatizer()

# --- 核心更新：根据侦查结果扩充黑名单 ---
STOP_WORDS = set([
    # 1. 之前发现的基础噪音
    'min', 'any', 'v8', 'vol', 'spread', 'reserved', 'half',
    'inch',

    # 2. 计量单位 (非常全)
    'oz', 'lb', 'lbs', 'ounce', 'ounces', 'gram', 'grams', 'kg', 'g',
    'teaspoon', 'tsp', 'tablespoon', 'tbsp', 'cup', 'cups',
    'quart', 'qt', 'pint', 'pt', 'gallon', 'gal', 'liter', 'litre',
    'ml', 'fluid', 'fl',

    # 3. 饮食与健康描述 (重灾区：low-fat, gluten-free, etc.)
    'low', 'fat', 'non', 'skim', 'part', 'reduced', 'less', 'sodium', 'free', 'gluten',
    'diet', 'lite', 'light', 'lean', 'extra', 'no', 'added', 'calorie',
    'sugar', 'sweetened', 'unsweetened', # 注意：sweet potato 的 sweet 不在这里，单独的 sweetened 要删

    # 4. 食材状态/预处理 (形容词)
    'chopped', 'diced', 'sliced', 'fresh', 'minced', 'large', 'small', 'medium', 'jumbo',
    'package', 'can', 'canned', 'container', 'box', 'bag', 'packet',
    'frozen', 'melted', 'beaten', 'ground', 'crushed', 'whole',
    'boneless', 'skinless', 'bone', 'skin', 'chunk', 'chunks',
    'shredded', 'grated', 'crumbled', 'flake', 'flaked', # [新增] 来自 shredded cheese
    'soft', 'softened', 'hard', 'boiled', # [新增] hard-boiled egg -> egg
    'active', 'dry', 'dried', # [新增] active yeast, dried basil -> yeast, basil

    # 5. 商业/营销/通用术语
    'brand', 'store', 'bought', 'ready', 'made', 'instant',
    'original', 'style', 'condensed', 'evaporated', # [新增] condensed milk -> milk
    'mix', 'mixture', 'blend', # [新增] seasoning mix -> seasoning
    'flavor', 'flavored', # [新增] chicken flavored -> chicken
    'old', 'fashioned', # [新增] old-fashioned oats -> oats
    'purpose', 'rising', # [新增] all-purpose, self-rising

    # 6. 逗号后面的动作 (动词)
    'drain', 'drained', 'undrain', 'undrained',
    'cook', 'cooked', 'cooking', 'uncooked',
    'rinse', 'rinsed', 'rins',
    'peel', 'peeled', 'devein', 'deveined', # [新增] 虾的处理
    'thaw', 'thawed',
    'cut', 'prepare', 'prepared', 'trimmed',
    'scrub', 'scrubbed', 'well', 'wash', 'washed',
    'squeeze', 'squeezed' # [新增] squeezed dry
])

def final_clean_pipeline(ingredient_list):
    clean_ingredients = []

    for item in ingredient_list:
        # 逗号变空格
        item = item.replace(',', ' ')

        # 字符标准化
        item = unicodedata.normalize('NFKD', item).encode('ascii', 'ignore').decode('utf-8')

        # 转小写
        text = item.lower()

        # 处理 half & half 为一个词
        if 'half & half' in text:
            text = text.replace('half & half', 'halfandhalf')

        # 处理连接符
        text = text.replace('-', ' ').replace('&', ' ')

        # 去数字、去标点
        text = re.sub(r'[^a-z]', ' ', text)

        # 单词级精修
        words = text.split()
        meaningful_words = []

        for w in words:
            # 过滤短词
            if len(w) < 2:
                continue

            # 过滤黑名单 (这里会杀掉 drain, soften, boneless 等)
            if w in STOP_WORDS:
                continue

            # 词形还原
            w = lemma.lemmatize(w)
            meaningful_words.append(w)

        # 组装
        if meaningful_words:
            clean_ingredients.append(' '.join(meaningful_words))

    return ' '.join(clean_ingredients)

# --- 执行清洗 ---
print("开始清洗训练集...")
train_df['ingredients_clean'] = train_df['ingredients'].apply(final_clean_pipeline)
print("开始清洗测试集...")
test_df['ingredients_clean'] = test_df['ingredients'].apply(final_clean_pipeline)

开始清洗训练集...
开始清洗测试集...


In [9]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    binary=True,          # 只要出现过就算，不管几次
    min_df=3,             # 出现少于 3 次的词直接扔掉
    max_features=5000     # 保留最重要的前 5000 个
)

X = vectorizer.fit_transform(train_df['ingredients_clean'])
X_submission = vectorizer.transform(test_df['ingredients_clean'])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_df['cuisine'])


In [14]:
# 1. 准备数据 (神经网络需要特定的格式)
# 必须把稀疏矩阵转为排序好的数组，或者直接支持稀疏输入的格式
# 这里为了简单，我们转为 dense (如果内存够的话)
# 警告：如果你 max_features 没设限，X.toarray() 可能会爆内存
# 建议：在 TF-IDF 处设置 max_features=5000 或者 8000
X_dense = X.toarray()
X_test_dense = X_submission.toarray() # 测试集也要转

# Label 需要 One-Hot 编码
from sklearn.preprocessing import LabelBinarizer
lb = LabelBinarizer()
y_onehot = lb.fit_transform(y)

# 2. 搭建网络结构 (专门处理文本分类的经典架构)
model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=(X_dense.shape[1],)), # 第一层
    layers.Dropout(0.5), # 防止过拟合 (关键！)
    layers.Dense(256, activation='relu'), # 第二层
    layers.Dropout(0.3),
    layers.Dense(20, activation='softmax') # 输出层 (20个菜系)
])

# 3. 编译
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 4. 训练
# epochs 不要太多，容易过拟合，用 EarlyStopping 最好
model.fit(X_dense, y_onehot, epochs=15, batch_size=128, validation_split=0.1)

# 5. 预测
y_pred_probs = model.predict(X_test_dense)

# 拿到最大概率的索引 (0, 1, 2...)
y_pred_indices = y_pred_probs.argmax(axis=1)

# --- 3. 还原标签 (关键修正步骤) ---
# 必须使用你最开始定义的 label_encoder
# 因为是它把 'mexican' 变成了数字，只有它能变回来
print("2. 正在将数字还原为菜系名称...")
y_pred_text = label_encoder.inverse_transform(y_pred_indices)

# --- 4. 打印检查 (确保是文字) ---
print("预测结果示例 (前5个):", y_pred_text[:5])
# 预期输出应该是: ['southern_us' 'mexican' 'italian' ...]
# 如果这里显示的是数字，说明 label_encoder 没定义对，但按你之前的代码应该是对的。

# --- 5. 生成 CSV 文件 ---
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'cuisine': y_pred_text
})

filename = 'submission_dnn_final.csv'
submission_df.to_csv(filename, index=False)

print("-" * 30)
print(f"✅ 成功！提交文件已生成: {filename}")
print(f"共预测了 {len(submission_df)} 条数据。")

Epoch 1/15
224/224 [==============================] - 2s 9ms/step - loss: 1.4739 - accuracy: 0.5802 - val_loss: 0.9113 - val_accuracy: 0.7291
Epoch 2/15
224/224 [==============================] - 2s 8ms/step - loss: 0.7844 - accuracy: 0.7673 - val_loss: 0.7984 - val_accuracy: 0.7662
Epoch 3/15
224/224 [==============================] - 2s 8ms/step - loss: 0.6035 - accuracy: 0.8170 - val_loss: 0.7795 - val_accuracy: 0.7696
Epoch 4/15
224/224 [==============================] - 2s 9ms/step - loss: 0.4820 - accuracy: 0.8514 - val_loss: 0.8108 - val_accuracy: 0.7696
Epoch 5/15
224/224 [==============================] - 2s 8ms/step - loss: 0.3842 - accuracy: 0.8828 - val_loss: 0.8523 - val_accuracy: 0.7621
Epoch 6/15
224/224 [==============================] - 2s 8ms/step - loss: 0.3032 - accuracy: 0.9043 - val_loss: 0.8890 - val_accuracy: 0.7586
Epoch 7/15
224/224 [==============================] - 2s 8ms/step - loss: 0.2405 - accuracy: 0.9252 - val_loss: 0.9601 - val_accuracy: 0.7586
Epoch 

In [15]:
# 切分数据
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 定义模型 (你可以在这里随意更换模型或调整参数)
model = LogisticRegression(C=3, max_iter=1000)

# model = LinearSVC(
#     C=0.5,
#     dual=False,
#     class_weight='balanced',
#     random_state=42
# )

from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# # --- 1. 定义你的复仇者联盟 (模型组合) ---
#
# # 钢铁侠：逻辑回归 (高准确率，大局观好)
# # 保持你之前 0.775 的那个配置
# clf_lr = LogisticRegression(
#     solver='lbfgs',
#     C=3,
#     max_iter=1000,
#     n_jobs=-1,
#     random_state=42
# )
#
# # 绿巨人：线性 SVM (高召回率，能通过 class_weight='balanced' 捞小样本)
# # 我们用 CalibratedClassifierCV 把它包起来，让它能输出概率
# svm_base = LinearSVC(
#     C=0.5,
#     dual=False,
#     class_weight='balanced',  # 保持这个激进的权重
#     random_state=42
# )
# clf_svc = CalibratedClassifierCV(svm_base)
#
# # --- 2. 组装投票器 ---
# # voting='soft' 意味着把两个模型预测的“概率值”加起来取平均，比单纯看“票数”更准
# voting_model = VotingClassifier(
#     estimators=[('lr', clf_lr), ('svc', clf_svc)],
#     voting='soft',
#     n_jobs=-1
# )
#
# # 1. 使用 X_train 和 y_train (而不是 X, y)
# print("正在使用训练集训练融合模型 (确保不偷看验证集)...")
# voting_model.fit(X_train, y_train)
#
# # 2. 现在再去预测验证集
# print("正在预测...")
# y_pred_vote = voting_model.predict(X_val)
#
# # 3. 打印真实报告
# print(classification_report(y_val, y_pred_vote, target_names=label_encoder.classes_))


# 训练
model.fit(X_train, y_train)

# 验证
y_pred = model.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)

print(f"当前模型验证集准确率: {val_acc:.4f}")
print(classification_report(y_val, y_pred, target_names=label_encoder.classes_))


# 1. 预测验证集
print("正在预测验证集...")
y_pred = model.predict(X_val)

# 2. 生成详细报告并转为 DataFrame 方便排序
report_dict = classification_report(y_val, y_pred, target_names=label_encoder.classes_, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

# 3. 按 F1-score 倒序排列，找出最差的 5 个
print("\n=== 表现最差的 5 个菜系 (The Troublemakers) ===")
print(report_df.sort_values(by='f1-score').head(5)[['precision', 'recall', 'f1-score', 'support']])

当前模型验证集准确率: 0.7769
              precision    recall  f1-score   support

   brazilian       0.74      0.48      0.58        71
     british       0.46      0.34      0.39       116
cajun_creole       0.75      0.69      0.72       227
     chinese       0.77      0.88      0.82       398
    filipino       0.78      0.57      0.66       134
      french       0.55      0.60      0.58       410
       greek       0.86      0.70      0.77       168
      indian       0.85      0.89      0.87       502
       irish       0.63      0.42      0.51       106
     italian       0.79      0.90      0.84      1293
    jamaican       0.90      0.62      0.74        96
    japanese       0.86      0.67      0.75       239
      korean       0.85      0.75      0.79       150
     mexican       0.89      0.92      0.91       964
    moroccan       0.79      0.68      0.73       121
     russian       0.51      0.27      0.35        75
 southern_us       0.72      0.79      0.75       736
     spa

In [38]:
from sklearn.model_selection import GridSearchCV

# 1. 定义要搜索的参数网格
# 我们要调整三个东西：
# - lr__C: 逻辑回归的强度
# - svc__base_estimator__C: SVM 的强度 (因为包了CalibratedClassifierCV，路径比较深)
# - weights: 投票权重 (是听LR的多一点，还是听SVM的多一点？)

params = {
    'lr__C': [1, 3, 5, 10],
    # 注意：因为 Voting -> Calibrated -> LinearSVC，参数名会很长，这是 sklearn 的命名规则
    # 如果报错，可以先只调 lr__C 和 weights
    'voting': ['soft'],
    'weights': [[1, 1], [2, 1], [1, 2], [3, 1]]
}

# 2. 建立网格搜索
# cv=3: 做3次交叉验证，确保结果稳
grid = GridSearchCV(estimator=voting_model, param_grid=params, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)

print("正在暴力搜索最佳参数... (去喝杯咖啡吧)")
grid.fit(X_train, y_train)

# 3. 公布最佳结果
print(f"\n最佳准确率: {grid.best_score_:.4f}")
print("最佳参数组合:", grid.best_params_)

# 4. 把最佳模型存下来
best_model = grid.best_estimator_

正在暴力搜索最佳参数... (去喝杯咖啡吧)
Fitting 3 folds for each of 16 candidates, totalling 48 fits

最佳准确率: 0.7600
最佳参数组合: {'lr__C': 3, 'voting': 'soft', 'weights': [1, 2]}


In [39]:

# --- 第一步：准备全量数据 ---
# X 和 y 是你最开始用 train_df 生成的完整特征矩阵和标签
# best_model 是刚才 GridSearch 跑出来的最佳模型 (包含了 C=3, weights=[1, 2] 等参数)
final_model = best_model

print(f"1. 正在使用全量训练集重新训练模型 (样本数: {X.shape[0]})...")
# 这一步非常关键！之前 GridSearch 只用了 2/3 的数据训练，现在我们要用 100% 的数据
final_model.fit(X, y)

# --- 第二步：准备测试集特征 ---
print("2. 正在处理测试集特征...")
# 警告：必须确保 test_df 已经执行过 final_clean_pipeline 清洗！
# 如果你重启过内核，请务必先运行一遍数据清洗的代码
# transform: 使用训练集学到的词典将测试集转为向量 (千万不要用 fit_transform)
X_test_final = vectorizer.transform(test_df['ingredients_clean'])

# --- 第三步：预测 ---
print("3. 正在预测 Kaggle 测试集...")
y_pred_indices = final_model.predict(X_test_final)

# --- 第四步：还原标签 (数字 -> 文本) ---
# 将 0, 1, 2 还原回 'italian', 'mexican' 等
y_pred_text = label_encoder.inverse_transform(y_pred_indices)

# --- 第五步：生成 CSV 文件 ---
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'cuisine': y_pred_text
})

filename = 'submission2.csv'
submission_df.to_csv(filename, index=False)

print("-" * 30)
print(f"提交文件已生成: {filename}")
print(f"共预测了 {len(submission_df)} 条数据。")

1. 正在使用全量训练集重新训练模型 (样本数: 31819)...
2. 正在处理测试集特征...
3. 正在预测 Kaggle 测试集...
------------------------------
✅ 成功！提交文件已生成: submission_voting_optimized.csv
共预测了 7955 条数据。
快去 Kaggle 提交，看看能不能冲进 Leaderboard 前列！🚀


In [ ]:
# https://www.kaggle.com/competitions/cuisine-prediction